# 05 - Análise visual e comparativa

A base da ANP cobre 2011 a 29/06/2026 e contempla João Pessoa, Cabedelo, Bayeux e Santa Rita neste recorte. Os resultados são históricos, não representam preços atuais em tempo real, e a pesquisa da ANP é amostral.

As séries históricas por município e combustível preservam a visão temporal existente. As comparações de postos e bairros, os rankings e os testes entre municípios usam uma semana de referência comum, escolhida como a mais recente em que os quatro municípios têm ao menos dois postos observados para cada combustível. Isso evita comparar períodos incompatíveis; a data de referência e a quantidade de postos por grupo são informadas nos resultados. Se não existir semana comum com amostra mínima, essas comparações são sinalizadas como indisponíveis.

As variações percentuais entre semanas são calculadas apenas entre observações separadas por sete dias. Períodos com intervalo maior permanecem identificados, mas não são apresentados como variação semanal consecutiva. O teste de Kruskal-Wallis compara os preços dos postos entre municípios nessa mesma semana e é exploratório, sem interpretação causal. Os preços sinalizados no Notebook 02 permanecem na análise. A paridade etanol/gasolina é mantida apenas como análise secundária.

In [8]:
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import kruskal

RAIZ = Path.cwd()
if not (RAIZ / "Dados").exists() and (RAIZ.parent / "Dados").exists():
    RAIZ = RAIZ.parent
assert (RAIZ / "Dados").exists(), "Confira a pasta aberta no VS Code."
PASTA_ANALITICO = RAIZ / "Dados/Analitico"
PASTA_FIGURAS = RAIZ / "figuras"
PASTA_TABELAS = RAIZ / "tabelas"
PASTA_FIGURAS.mkdir(parents=True, exist_ok=True)
PASTA_TABELAS.mkdir(parents=True, exist_ok=True)

plt.style.use("seaborn-v0_8-whitegrid")
sns.set_context("notebook")

precos = pd.read_parquet(PASTA_ANALITICO / "precos_municipio_semana.parquet")
metricas = pd.read_parquet(PASTA_ANALITICO / "metricas.parquet")
posto = pd.read_parquet(PASTA_ANALITICO / "posto_semana.parquet")
dispersao = pd.read_parquet(PASTA_ANALITICO / "metricas_dispersao.parquet")
ranking_postos = pd.read_parquet(PASTA_ANALITICO / "ranking_postos_semana.parquet")
metricas_posto = pd.read_parquet(PASTA_ANALITICO / "metricas_posto_semana.parquet")
metricas_municipio = pd.read_parquet(PASTA_ANALITICO / "metricas_municipio_semana.parquet")

colunas_ranking = {
    "semana", "municipio", "bairro", "produto", "id_posto", "revenda",
    "logradouro", "numero", "preco_medio", "rank_mais_barato", "rank_mais_caro"
}
ausentes = colunas_ranking.difference(ranking_postos.columns)
if ausentes:
    raise ValueError(f"O ranking do Notebook 04 não contém as colunas esperadas: {sorted(ausentes)}")

print("Período analisado:", precos["semana"].min(), "até", precos["semana"].max())
print("Municípios:", ", ".join(sorted(precos["municipio"].unique())))
print("Postos com CNPJ válido:", posto["id_posto"].nunique())
print("Linhas no ranking semanal:", len(ranking_postos))
print("Observação: os dados são históricos, não preços em tempo real.")

Período analisado: 2011-01-03 00:00:00 até 2026-06-29 00:00:00
Municípios: BAYEUX, CABEDELO, JOAO PESSOA, SANTA RITA
Postos com CNPJ válido: 205
Linhas no ranking semanal: 69285
Observação: os dados são históricos, não preços em tempo real.


In [9]:
for municipio in sorted(precos["municipio"].dropna().unique()):
    dados = precos.loc[precos["municipio"] == municipio].sort_values("semana")
    fig, ax = plt.subplots(figsize=(12, 5))
    ax.plot(dados["semana"], dados["GASOLINA"], label="Gasolina", linewidth=1.8)
    ax.plot(dados["semana"], dados["ETANOL"], label="Etanol", linewidth=1.8)
    ax.set(title=f"Preços semanais históricos - {municipio}", xlabel="Semana", ylabel="Preço (R$/L)")
    ax.legend()
    fig.tight_layout()
    fig.savefig(PASTA_FIGURAS / f"precos_{municipio.lower().replace(' ', '_')}.png", dpi=300)
    plt.close(fig)

In [10]:
fig, ax = plt.subplots(figsize=(12, 6))
for municipio in sorted(metricas["municipio"].dropna().unique()):
    dados = metricas.loc[metricas["municipio"] == municipio].sort_values("semana")
    ax.plot(dados["semana"], dados["razao"], label=municipio, linewidth=1.5)
ax.axhline(0.70, color="firebrick", linestyle="--", linewidth=1.5, label="Regra indicativa 0,70")
ax.set(title="Razão semanal entre etanol e gasolina", xlabel="Semana", ylabel="Preço do etanol / preço da gasolina")
ax.legend(bbox_to_anchor=(1.02, 1), loc="upper left")
fig.tight_layout()
fig.savefig(PASTA_FIGURAS / "razao_por_municipio.png", dpi=300, bbox_inches="tight")
plt.close(fig)

In [11]:
fig, ax = plt.subplots(figsize=(10, 6))
sns.boxplot(data=metricas, x="municipio", y="razao", hue="municipio", legend=False, ax=ax)
ax.axhline(0.70, color="firebrick", linestyle="--", linewidth=1.5)
ax.set(title="Distribuição histórica da razão etanol/gasolina", xlabel="Município", ylabel="Etanol / gasolina")
fig.tight_layout()
fig.savefig(PASTA_FIGURAS / "boxplot_razao_por_municipio.png", dpi=300)
plt.close(fig)

In [12]:
mensal = precos.copy()
mensal["data"] = pd.to_datetime(mensal["semana"])
mensal["ano"] = mensal["data"].dt.year
mensal["mes_numero"] = mensal["data"].dt.month
mensal = (
    mensal.groupby(["municipio", "ano", "mes_numero"], as_index=False)[["ETANOL", "GASOLINA"]]
    .mean()
)

# Centraliza cada município-ano para separar o padrão sazonal da tendência de nível dos preços.
for combustivel in ["ETANOL", "GASOLINA"]:
    mensal[f"{combustivel}_desvio_anual"] = mensal.groupby(["municipio", "ano"])[combustivel].transform(
        lambda valores: valores - valores.mean()
    )
sazonalidade = (
    mensal.groupby("mes_numero", as_index=False)[["ETANOL_desvio_anual", "GASOLINA_desvio_anual"]]
    .mean()
)

fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(sazonalidade["mes_numero"], sazonalidade["GASOLINA_desvio_anual"], marker="o", label="Gasolina")
ax.plot(sazonalidade["mes_numero"], sazonalidade["ETANOL_desvio_anual"], marker="o", label="Etanol")
ax.axhline(0, color="black", linewidth=1)
ax.set(xticks=range(1, 13), xlabel="Mês do ano", ylabel="Desvio da média anual (R$/L)", title="Sazonalidade após centralização por município e ano")
ax.legend()
fig.tight_layout()
fig.savefig(PASTA_FIGURAS / "sazonalidade_precos.png", dpi=300)
plt.close(fig)
display(sazonalidade)

,mes_numero,ETANOL_desvio_anual,GASOLINA_desvio_anual
0,1,-0.185933,-0.132766
1,2,-0.129101,-0.087151
2,3,-0.057418,-0.01652
3,4,0.016896,0.003864
4,5,0.072585,0.012122
5,6,0.101834,0.038364
6,7,0.109074,-0.010709
7,8,0.09641,0.003007
8,9,0.050263,0.012209
9,10,-0.00241,0.048185


In [13]:
# CV histórico: descreve a série, não substitui a comparação de período comum abaixo.
cv_resumo = dispersao.groupby(["municipio", "produto"], as_index=False)["cv_pct"].mean()
fig, ax = plt.subplots(figsize=(10, 5))
sns.barplot(data=cv_resumo, x="municipio", y="cv_pct", hue="produto", ax=ax)
ax.set(title="CV semanal médio na série histórica", xlabel="Município", ylabel="CV médio (%)")
fig.tight_layout()
fig.savefig(PASTA_FIGURAS / "cv_precos_por_municipio.png", dpi=300)
plt.close(fig)

fig, ax = plt.subplots(figsize=(12, 6))
for (municipio, produto), grupo in dispersao.groupby(["municipio", "produto"]):
    grupo = grupo.sort_values("semana")
    ax.plot(grupo["semana"], grupo["desvio_padrao"], label=f"{municipio} - {produto}", linewidth=1)
ax.set(title="Dispersão semanal dos preços entre postos", xlabel="Semana", ylabel="Desvio-padrão (R$/L)")
ax.legend(bbox_to_anchor=(1.02, 1), loc="upper left")
fig.tight_layout()
fig.savefig(PASTA_FIGURAS / "dispersao_precos.png", dpi=300, bbox_inches="tight")
plt.close(fig)

# Define um recorte comum para comparar postos, bairros e municípios sem misturar anos.
municipios_comparacao = sorted(precos["municipio"].dropna().unique())
combustiveis = sorted(ranking_postos["produto"].dropna().unique())
grupos_esperados = pd.MultiIndex.from_product(
    [municipios_comparacao, combustiveis], names=["municipio", "produto"]
)
contagem_grupos = (
    ranking_postos.groupby(["semana", "municipio", "produto"])["id_posto"]
    .nunique()
    .unstack(["municipio", "produto"])
    .reindex(columns=grupos_esperados)
)
semanas_completas = contagem_grupos.dropna(how="any")
semanas_suficientes = semanas_completas.loc[(semanas_completas >= 2).all(axis=1)]
semana_referencia = semanas_suficientes.index.max() if not semanas_suficientes.empty else None

if semana_referencia is None:
    recorte_comum = ranking_postos.iloc[0:0].copy()
    print("Limitação: não há semana com pelo menos dois postos em todos os municípios e combustíveis.")
else:
    recorte_comum = ranking_postos.loc[ranking_postos["semana"] == semana_referencia].copy()
    print("Semana comum de referência:", semana_referencia.date())
    print("Postos por município e combustível:")
    display(recorte_comum.groupby(["municipio", "produto"])["id_posto"].nunique().unstack())

recorte_comum.to_csv(
    PASTA_TABELAS / "ranking_postos_semana_comum.csv", index=False, sep=";", decimal=","
)
colunas_resumo = [
    "municipio", "produto", "preco_medio", "preco_min", "preco_max",
    "desvio_padrao", "amplitude", "cv_pct", "postos", "bairros"
]
if recorte_comum.empty:
    resumo_semana_comum = pd.DataFrame(columns=colunas_resumo)
else:
    resumo_semana_comum = (
        recorte_comum.groupby(["municipio", "produto"], as_index=False)
        .agg(
            preco_medio=("preco_medio", "mean"),
            preco_min=("preco_medio", "min"),
            preco_max=("preco_medio", "max"),
            desvio_padrao=("preco_medio", "std"),
            postos=("id_posto", "nunique"),
            bairros=("bairro", "nunique")
        )
    )
    resumo_semana_comum["amplitude"] = resumo_semana_comum["preco_max"] - resumo_semana_comum["preco_min"]
    resumo_semana_comum["cv_pct"] = 100 * resumo_semana_comum["desvio_padrao"] / resumo_semana_comum["preco_medio"]
    resumo_semana_comum = resumo_semana_comum[colunas_resumo]
resumo_semana_comum["semana_referencia"] = semana_referencia
resumo_semana_comum.to_csv(
    PASTA_TABELAS / "resumo_precos_semana_comum.csv", index=False, sep=";", decimal=","
)

if not recorte_comum.empty:
    # Intervalo mínimo-máximo e média por município/combustível na semana comum.
    fig, axes = plt.subplots(2, 4, figsize=(18, 5.5), sharex="col")
    for linha, produto in enumerate(combustiveis):
        for coluna, municipio in enumerate(municipios_comparacao):
            ax = axes[linha, coluna]
            dados = resumo_semana_comum.loc[
                (resumo_semana_comum["municipio"] == municipio)
                & (resumo_semana_comum["produto"] == produto)
            ]
            ax.set_title(f"{municipio} · {produto.title()}")
            if dados.empty:
                ax.text(0.5, 0.5, "Sem amostra suficiente", ha="center", va="center", transform=ax.transAxes)
                continue
            item = dados.iloc[0]
            media = item["preco_medio"]
            ax.errorbar(
                media, 0,
                xerr=[[media - item["preco_min"]], [item["preco_max"] - media]],
                fmt="o", color="#176B55", ecolor="#D56A45", capsize=6, markersize=7
            )
            ax.set_yticks([])
            ax.set_ylim(-0.22, 0.22)
            ax.set_xlabel("Preço (R$/L)")
            ax.text(
                0.03, 0.94,
                f"Mín. {item['preco_min']:.3f} · Máx. {item['preco_max']:.3f}\nCV {item['cv_pct']:.1f}% · n={int(item['postos'])}",
                transform=ax.transAxes, va="top", fontsize=8
            )
    fig.suptitle(f"Média e amplitude dos preços por posto · {semana_referencia:%d/%m/%Y}")
    fig.tight_layout(rect=[0, 0, 1, 0.92])
    fig.savefig(PASTA_FIGURAS / "precos_resumo_semana_comum.png", dpi=300, bbox_inches="tight")
    plt.close(fig)

    # Cada ponto é um posto; os bairros são comparados na mesma semana.
    fig, axes = plt.subplots(2, 4, figsize=(22, 9), sharey="row")
    for linha, produto in enumerate(combustiveis):
        for coluna, municipio in enumerate(municipios_comparacao):
            ax = axes[linha, coluna]
            dados = recorte_comum.loc[
                (recorte_comum["municipio"] == municipio)
                & (recorte_comum["produto"] == produto)
            ].copy()
            ax.set_title(f"{municipio} · {produto.title()}")
            if dados.empty:
                ax.text(0.5, 0.5, "Sem amostra", ha="center", va="center", transform=ax.transAxes)
                continue
            ordem_bairros = dados.groupby("bairro")["preco_medio"].median().sort_values().index.tolist()
            sns.boxplot(
                data=dados, x="bairro", y="preco_medio", order=ordem_bairros,
                color="#DCE9E1", fliersize=0, width=0.6, ax=ax
            )
            sns.stripplot(
                data=dados, x="bairro", y="preco_medio", order=ordem_bairros,
                color="#D56A45", jitter=0.18, size=3.5, ax=ax
            )
            ax.tick_params(axis="x", rotation=85, labelsize=7)
            ax.set_xlabel("Bairro")
            ax.set_ylabel("Preço por posto (R$/L)")
    fig.suptitle(f"Distribuição dos preços dos postos por bairro · {semana_referencia:%d/%m/%Y}")
    fig.tight_layout(rect=[0, 0, 1, 0.93])
    fig.savefig(PASTA_FIGURAS / "precos_postos_por_bairro_semana_comum.png", dpi=300, bbox_inches="tight")
    plt.close(fig)

    # Até três posições de cada ponta em cada município, com valor identificado.
    fig, axes = plt.subplots(2, 2, figsize=(18, 11))
    for coluna, produto in enumerate(combustiveis):
        menor = recorte_comum.loc[
            (recorte_comum["produto"] == produto) & (recorte_comum["rank_mais_barato"] <= 3)
        ].sort_values("preco_medio").copy()
        maior = recorte_comum.loc[
            (recorte_comum["produto"] == produto) & (recorte_comum["rank_mais_caro"] <= 3)
        ].sort_values("preco_medio", ascending=False).copy()
        for linha, (dados, titulo) in enumerate([
            (menor, "Até três menores preços por município"),
            (maior, "Até três maiores preços por município")
        ]):
            ax = axes[linha, coluna]
            ax.set_title(f"{titulo} · {produto.title()}")
            if dados.empty:
                ax.text(0.5, 0.5, "Sem amostra", ha="center", va="center", transform=ax.transAxes)
                continue
            dados["rotulo"] = dados.apply(
                lambda posto: f"{posto['municipio'][:7]} · {str(posto['bairro'])[:12]} · {str(posto['revenda'])[:18]}",
                axis=1
            )
            barras = ax.barh(dados["rotulo"], dados["preco_medio"], color="#176B55")
            ax.bar_label(barras, fmt="R$ %.2f", padding=3, fontsize=7)
            ax.invert_yaxis()
            ax.set_xlabel("Preço médio por posto (R$/L)")
            ax.tick_params(axis="y", labelsize=7)
            ax.margins(x=0.18)
    fig.suptitle(f"Rankings de postos na mesma semana · {semana_referencia:%d/%m/%Y}")
    fig.tight_layout(rect=[0, 0, 1, 0.93])
    fig.savefig(PASTA_FIGURAS / "ranking_precos_semana_comum.png", dpi=300, bbox_inches="tight")
    plt.close(fig)

# Variações de sete dias: postos sem semana adjacente não entram nesta visualização.
variacao_municipal = metricas_municipio.loc[
    metricas_municipio["comparacao_semanal_consecutiva"]
].copy()
variacao_postos = (
    metricas_posto.loc[metricas_posto["comparacao_semanal_consecutiva"]]
    .groupby(["semana", "municipio", "produto"], as_index=False)
    .agg(variacao_pct_postos=("variacao_pct", "median"), postos_comparados=("id_posto", "nunique"))
)
variacao_plot = variacao_municipal.merge(
    variacao_postos, on=["semana", "municipio", "produto"], how="left", validate="one_to_one"
)
fig, axes = plt.subplots(2, 4, figsize=(20, 9), sharey="row")
for linha, produto in enumerate(combustiveis):
    for coluna, municipio in enumerate(municipios_comparacao):
        ax = axes[linha, coluna]
        dados = variacao_plot.loc[
            (variacao_plot["municipio"] == municipio)
            & (variacao_plot["produto"] == produto)
        ].sort_values("semana")
        ax.set_title(f"{municipio} · {produto.title()}")
        if dados.empty:
            ax.text(0.5, 0.5, "Sem semanas consecutivas", ha="center", va="center", transform=ax.transAxes)
            continue
        ax.plot(dados["semana"], dados["variacao_pct"], color="#176B55", linewidth=0.8, label="Média municipal")
        ax.plot(dados["semana"], dados["variacao_pct_postos"], color="#D56A45", linewidth=0.8, alpha=0.8, label="Mediana dos postos")
        ax.axhline(0, color="#555555", linewidth=0.7)
        ax.set_xlabel("Semana")
        ax.set_ylabel("Variação semanal (%)")
        ax.legend(fontsize=7)
fig.suptitle("Variação percentual entre semanas consecutivas")
fig.tight_layout()
fig.savefig(PASTA_FIGURAS / "variacao_percentual_semanal.png", dpi=300, bbox_inches="tight")
plt.close(fig)

# Kruskal-Wallis entre municípios usando apenas postos da mesma semana de referência.
resultados_teste = []
if semana_referencia is not None:
    for produto in combustiveis:
        dados_produto = recorte_comum.loc[recorte_comum["produto"] == produto]
        grupos = {
            municipio: grupo["preco_medio"].to_numpy()
            for municipio, grupo in dados_produto.groupby("municipio")
            if grupo["id_posto"].nunique() >= 2
        }
        if len(grupos) < 2:
            print(f"Kruskal-Wallis não aplicado a {produto}: menos de dois municípios com ao menos dois postos.")
            continue
        teste = kruskal(*grupos.values())
        quantidade = sum(len(valores) for valores in grupos.values())
        k = len(grupos)
        resultados_teste.append({
            "semana_referencia": semana_referencia,
            "produto": produto,
            "municipios": k,
            "postos": quantidade,
            "postos_por_municipio": " | ".join(f"{municipio}: {len(valores)}" for municipio, valores in grupos.items()),
            "estatistica_H": teste.statistic,
            "p_valor": teste.pvalue,
            "epsilon_quadrado": max(0, (teste.statistic - k + 1) / (quantidade - k))
        })

colunas_teste = [
    "semana_referencia", "produto", "municipios", "postos", "postos_por_municipio",
    "estatistica_H", "p_valor", "epsilon_quadrado", "p_ajustado_bonferroni", "significativo_05"
]
testes_kruskal = pd.DataFrame(resultados_teste)
if not testes_kruskal.empty:
    testes_kruskal["p_ajustado_bonferroni"] = (
        testes_kruskal["p_valor"] * len(testes_kruskal)
    ).clip(upper=1)
    testes_kruskal["significativo_05"] = testes_kruskal["p_ajustado_bonferroni"] < 0.05
else:
    testes_kruskal = pd.DataFrame(columns=colunas_teste)
testes_kruskal.to_csv(
    PASTA_TABELAS / "testes_kruskal_municipios.csv", index=False, sep=";", decimal=","
)

if not testes_kruskal.empty:
    print("Kruskal-Wallis: comparação entre municípios na mesma semana de referência.")
    print("Ajuste de Bonferroni aplicado aos testes disponíveis por combustível.")
    display(testes_kruskal)
else:
    print("Nenhum teste estatístico foi sustentado pelos grupos com dados comparáveis.")

display(resumo_semana_comum)
if not recorte_comum.empty:
    display(recorte_comum.sort_values(["municipio", "produto", "rank_mais_barato"]).head(12))

Semana comum de referência: 2017-07-24
Postos por município e combustível:


produto,ETANOL,GASOLINA
municipio,,
BAYEUX,6,7
CABEDELO,5,6
JOAO PESSOA,17,16
SANTA RITA,3,3


Kruskal-Wallis: comparação entre municípios na mesma semana de referência.
Ajuste de Bonferroni aplicado aos testes disponíveis por combustível.


,semana_referencia,produto,municipios,postos,postos_por_municipio,estatistica_H,p_valor,epsilon_quadrado,p_ajustado_bonferroni,significativo_05
0,2017-07-24,ETANOL,4,31,BAYEUX: 6 | CABEDELO: 5 | JOAO PESSOA: 17 | SA...,2.831622,0.418321,0,0.836642,False
1,2017-07-24,GASOLINA,4,32,BAYEUX: 7 | CABEDELO: 6 | JOAO PESSOA: 16 | SA...,2.886195,0.409507,0,0.819013,False


,municipio,produto,preco_medio,preco_min,preco_max,desvio_padrao,amplitude,cv_pct,postos,bairros,semana_referencia
0,BAYEUX,ETANOL,3.008333,2.74,3.29,0.218212,0.55,7.253599,6,5,2017-07-24
1,BAYEUX,GASOLINA,3.592429,3.16,3.85,0.226194,0.69,6.296403,7,6,2017-07-24
2,CABEDELO,ETANOL,3.0096,2.769,3.35,0.237435,0.581,7.889249,5,5,2017-07-24
3,CABEDELO,GASOLINA,3.579667,3.269,3.89,0.247306,0.621,6.908633,6,6,2017-07-24
4,JOAO PESSOA,ETANOL,2.951118,2.749,3.089,0.090867,0.34,3.07908,17,14,2017-07-24
5,JOAO PESSOA,GASOLINA,3.61975,3.49,3.689,0.047606,0.199,1.315174,16,14,2017-07-24
6,SANTA RITA,ETANOL,3.029333,2.999,3.09,0.052539,0.091,1.734338,3,2,2017-07-24
7,SANTA RITA,GASOLINA,3.556,3.479,3.599,0.066836,0.12,1.879517,3,2,2017-07-24


,semana,estado,municipio,produto,id_posto,cnpj_revenda,preco_medio,preco_min,preco_max,desvio_coletas,...,bandeira,amplitude,postos_no_periodo,rank_mais_barato,rank_mais_caro,preco_min_entre_postos,preco_max_entre_postos,diferenca_preco_entre_postos,comparacao_suficiente,observacao_comparacao
39243,2017-07-24,PB,BAYEUX,ETANOL,01420327000185,01420327000185,2.74,2.74,2.74,<NA>,...,BRANCA,0.0,6,1.0,6.0,2.74,3.29,0.55,True,
39244,2017-07-24,PB,BAYEUX,ETANOL,09230509000103,09230509000103,2.89,2.89,2.89,<NA>,...,IPIRANGA,0.0,6,2.0,4.0,2.74,3.29,0.55,True,
39245,2017-07-24,PB,BAYEUX,ETANOL,15386712000168,15386712000168,2.89,2.89,2.89,<NA>,...,BRANCA,0.0,6,2.0,4.0,2.74,3.29,0.55,True,
39246,2017-07-24,PB,BAYEUX,ETANOL,10542290000159,10542290000159,2.99,2.99,2.99,<NA>,...,BRANCA,0.0,6,4.0,3.0,2.74,3.29,0.55,True,
39247,2017-07-24,PB,BAYEUX,ETANOL,09359690000144,09359690000144,3.25,3.25,3.25,<NA>,...,PETROBRAS DISTRIBUIDORA S.A.,0.0,6,5.0,2.0,2.74,3.29,0.55,True,
39248,2017-07-24,PB,BAYEUX,ETANOL,24206617002170,24206617002170,3.29,3.29,3.29,<NA>,...,PETROBRAS DISTRIBUIDORA S.A.,0.0,6,6.0,1.0,2.74,3.29,0.55,True,
39249,2017-07-24,PB,BAYEUX,GASOLINA,01420327000185,01420327000185,3.16,3.16,3.16,<NA>,...,BRANCA,0.0,7,1.0,7.0,3.16,3.85,0.69,True,
39250,2017-07-24,PB,BAYEUX,GASOLINA,09230509000103,09230509000103,3.499,3.499,3.499,<NA>,...,IPIRANGA,0.0,7,2.0,6.0,3.16,3.85,0.69,True,
39251,2017-07-24,PB,BAYEUX,GASOLINA,10542290000159,10542290000159,3.589,3.589,3.589,<NA>,...,BRANCA,0.0,7,3.0,5.0,3.16,3.85,0.69,True,
39252,2017-07-24,PB,BAYEUX,GASOLINA,15386712000168,15386712000168,3.59,3.59,3.59,<NA>,...,BRANCA,0.0,7,4.0,4.0,3.16,3.85,0.69,True,
